# Section 1: Introduction & Model Overview

## Language Models & GPT-2

**Language Models** are probabilistic models that assign a probability to sequences of words. They are the backbone of modern NLP tasks such as text generation, translation, summarisation, and question answering.

### Why GPT-2?
- GPT-2 (Generative Pre-trained Transformer 2) was released by OpenAI in 2019.
- It is an **autoregressive** model — it predicts the next token given all previous tokens.
- Despite being several generations old, GPT-2 (small, 117 M parameters) is an excellent teaching model:
  - Small enough to run on a CPU or modest GPU
  - Large enough to produce coherent text
  - Fully open-source via HuggingFace Hub

### GPT-2 Architecture (small variant)
| Property | Value |
|---|---|
| Architecture | Transformer decoder (autoregressive) |
| Layers | 12 |
| Hidden size | 768 |
| Attention heads | 12 |
| Parameters | ~117 M |
| Context window | 1 024 tokens |
| Vocabulary size | 50 257 BPE tokens |

### Notebook Outline
1. Introduction & Model Overview
2. Environment Setup & Library Installation
3. Loading Pre-trained GPT-2
4. Text Generation with Different Decoding Strategies
5. Fine-tuning GPT-2 on a Custom Dataset
6. Performance Analysis & Evaluation Metrics
7. Visualisations
8. Attention Visualisation
9. Conclusions & Discussion

# Section 2: Environment Setup & Library Installation

In [ ]:
!pip install transformers torch datasets matplotlib seaborn nltk scikit-learn

In [ ]:
# --- Imports ---
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from datasets import load_dataset
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print('All libraries imported successfully.')

In [ ]:
# --- Reproducibility & Device ---
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Section 3: Loading Pre-trained GPT-2

In [ ]:
# --- Load tokenizer and model ---
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')
model = model.to(device)
model.eval()
print('GPT-2 loaded and set to eval mode.')

In [ ]:
# --- Model summary ---
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters   : {total_params:,}')
print(f'Trainable params   : {trainable_params:,}')
print(f'\nModel config:')
print(f'  n_layer          : {model.config.n_layer}')
print(f'  n_head           : {model.config.n_head}')
print(f'  n_embd           : {model.config.n_embd}')
print(f'  vocab_size        : {model.config.vocab_size}')
print(f'  n_positions       : {model.config.n_positions}')

In [ ]:
# --- Tokenisation demo ---
example_text = 'Artificial intelligence is transforming the world.'
token_ids = tokenizer.encode(example_text)
tokens = tokenizer.convert_ids_to_tokens(token_ids)
decoded = tokenizer.decode(token_ids)

print(f'Original text : {example_text}')
print(f'Token IDs     : {token_ids}')
print(f'Tokens        : {tokens}')
print(f'Decoded text  : {decoded}')

# Section 4: Text Generation with Different Decoding Strategies

GPT-2 supports several **decoding strategies** that control the trade-off between diversity and coherence:

| Strategy | Description |
|---|---|
| Greedy | Always pick the highest-probability next token |
| Beam search | Maintain top-N sequences simultaneously |
| Top-k | Sample from the top-k most likely tokens |
| Top-p (nucleus) | Sample from the smallest set of tokens whose cumulative probability ≥ p |
| Temperature | Scale the logits before softmax to control randomness |

In [ ]:
# --- Generation helper ---
def generate_text(prompt, strategy, **kwargs):
    """Generate text from a prompt using the specified decoding strategy."""
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        if strategy == 'greedy':
            output = model.generate(input_ids, max_length=100, do_sample=False)
        elif strategy == 'beam_search':
            output = model.generate(
                input_ids, max_length=100, num_beams=5, no_repeat_ngram_size=2
            )
        elif strategy == 'top_k':
            output = model.generate(
                input_ids, max_length=100, do_sample=True, top_k=50
            )
        elif strategy == 'top_p':
            output = model.generate(
                input_ids, max_length=100, do_sample=True, top_p=0.95
            )
        elif strategy == 'temperature':
            temp = kwargs.get('temperature', 1.0)
            output = model.generate(
                input_ids, max_length=100, do_sample=True, temperature=temp
            )
        else:
            raise ValueError(f'Unknown strategy: {strategy}')
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
# --- Compare strategies across prompts ---
prompts = [
    'Artificial intelligence will transform',
    'The future of machine learning is',
    'Once upon a time in a land far away',
    'The most important scientific discovery',
]
strategies = ['greedy', 'beam_search', 'top_k', 'top_p']

results = []
for prompt in prompts:
    for strategy in strategies:
        generated = generate_text(prompt, strategy)
        results.append({'Prompt': prompt, 'Strategy': strategy, 'Generated Text': generated})

df_results = pd.DataFrame(results)
pd.set_option('display.max_colwidth', 120)
display(df_results)

In [ ]:
# --- Temperature scaling demo ---
prompt = 'The future of machine learning is'
temperatures = [0.3, 0.7, 1.0, 1.5]

print('Temperature scaling effect:')
print('=' * 80)
for temp in temperatures:
    text = generate_text(prompt, 'temperature', temperature=temp)
    print(f'\nTemperature = {temp}:')
    print(text)

# Section 5: Fine-tuning GPT-2 on a Custom Dataset

We fine-tune GPT-2 on the **WikiText-2** dataset — a collection of high-quality Wikipedia articles commonly used as a language modelling benchmark.

In [ ]:
# --- Load dataset ---
dataset = load_dataset('wikitext', 'wikitext-2-raw-v1')
print(dataset)
print('\nSample training text:')
print(dataset['train'][5]['text'])

In [ ]:
# --- Tokenise dataset ---
block_size = 128
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=block_size,
        padding='max_length',
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text'],
)
tokenized_dataset.set_format('torch')
print('Dataset tokenised.')
print(tokenized_dataset)

In [ ]:
# --- Fine-tune ---
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Use a small subset to keep training time manageable in this demo
small_train = tokenized_dataset['train'].select(range(500))

training_args = TrainingArguments(
    output_dir='./gpt2-finetuned',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    save_steps=500,
    logging_dir='./logs',
    report_to='none',
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    data_collator=data_collator,
)

trainer.train()

In [ ]:
# --- Plot training loss ---
log_history = trainer.state.log_history
train_logs = [entry for entry in log_history if 'loss' in entry]

steps = [entry['step'] for entry in train_logs]
losses = [entry['loss'] for entry in train_logs]

sns.set_style('whitegrid')
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(steps, losses, marker='o', color='steelblue', linewidth=2)
ax.set_title('Training Loss Curve', fontsize=14)
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
plt.tight_layout()
plt.show()

In [ ]:
# --- Compare base vs fine-tuned generation ---
# Reload the base model for comparison
base_model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
base_model.eval()

test_prompt = 'The history of science shows that'

def gen_with(mdl, prompt, max_length=80):
    ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = mdl.generate(ids, max_length=max_length, do_sample=True, top_p=0.95)
    return tokenizer.decode(out[0], skip_special_tokens=True)

print('Base GPT-2:')
print(gen_with(base_model, test_prompt))
print('\nFine-tuned GPT-2:')
print(gen_with(model, test_prompt))

# Section 6: Performance Analysis & Evaluation Metrics

We evaluate the models using:
- **Perplexity** — lower is better; measures how well the model predicts a text sample
- **BLEU score** — measures n-gram overlap between generated and reference text
- **Type-token ratio (TTR)** — measures lexical diversity of generated text

In [ ]:
# --- Perplexity ---
import math

def calculate_perplexity(mdl, tokenizer, text, device):
    """Calculate perplexity of a model on a given text string."""
    encodings = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    input_ids = encodings.input_ids.to(device)
    with torch.no_grad():
        outputs = mdl(input_ids, labels=input_ids)
        loss = outputs.loss
    return math.exp(loss.item())

sample_text = ' '.join(
    [ex['text'] for ex in dataset['validation'].select(range(20)) if ex['text'].strip()]
)

base_ppl = calculate_perplexity(base_model, tokenizer, sample_text, device)
ft_ppl = calculate_perplexity(model, tokenizer, sample_text, device)

print(f'Base GPT-2 perplexity     : {base_ppl:.2f}')
print(f'Fine-tuned GPT-2 ppl      : {ft_ppl:.2f}')

In [ ]:
# --- BLEU score ---
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk

ref_text = 'Artificial intelligence will transform how we live and work in profound ways.'
gen_text_base = gen_with(base_model, 'Artificial intelligence will transform')
gen_text_ft   = gen_with(model,      'Artificial intelligence will transform')

reference  = [nltk.word_tokenize(ref_text.lower())]
hyp_base   = nltk.word_tokenize(gen_text_base.lower())
hyp_ft     = nltk.word_tokenize(gen_text_ft.lower())

sf = SmoothingFunction().method1
bleu_base = sentence_bleu(reference, hyp_base, smoothing_function=sf)
bleu_ft   = sentence_bleu(reference, hyp_ft,   smoothing_function=sf)

print(f'Base GPT-2 BLEU score   : {bleu_base:.4f}')
print(f'Fine-tuned GPT-2 BLEU   : {bleu_ft:.4f}')

In [ ]:
# --- Type-token ratio (lexical diversity) ---
def type_token_ratio(text):
    tokens = nltk.word_tokenize(text.lower())
    if not tokens:
        return 0.0
    return len(set(tokens)) / len(tokens)

ttr_base = type_token_ratio(gen_text_base)
ttr_ft   = type_token_ratio(gen_text_ft)

print(f'Base GPT-2 TTR   : {ttr_base:.4f}')
print(f'Fine-tuned TTR   : {ttr_ft:.4f}')

In [ ]:
# --- Summary metrics DataFrame ---
summary = pd.DataFrame({
    'Model': ['Base GPT-2', 'Fine-tuned GPT-2'],
    'Perplexity': [base_ppl, ft_ppl],
    'BLEU Score': [bleu_base, bleu_ft],
    'Type-Token Ratio': [ttr_base, ttr_ft],
})
display(summary)

# Section 7: Visualisations

In [ ]:
# --- Top-10 next token probability bar chart ---
sns.set_style('whitegrid')

prompt = 'The future of artificial intelligence'
input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

with torch.no_grad():
    outputs = base_model(input_ids)
    next_token_logits = outputs.logits[0, -1, :]
    probs = torch.softmax(next_token_logits, dim=-1)
    top_k_probs, top_k_ids = torch.topk(probs, 10)

top_tokens = [tokenizer.decode([idx.item()]).strip() for idx in top_k_ids]
top_probs  = top_k_probs.cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top_tokens[::-1], top_probs[::-1], color=sns.color_palette('Blues_d', 10))
ax.set_xlabel('Probability')
ax.set_title(f'Top-10 Next Token Probabilities\nPrompt: "{prompt}"', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Perplexity comparison bar chart ---
fig, ax = plt.subplots(figsize=(6, 4))
models = ['Base GPT-2', 'Fine-tuned GPT-2']
ppls   = [base_ppl, ft_ppl]
colors = ['#4C72B0', '#55A868']
ax.bar(models, ppls, color=colors, edgecolor='black', width=0.4)
ax.set_ylabel('Perplexity (lower is better)')
ax.set_title('Perplexity: Base vs Fine-tuned GPT-2')
for i, v in enumerate(ppls):
    ax.text(i, v + 0.3, f'{v:.2f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Temperature effect on softmax distribution ---
logits = next_token_logits.cpu().numpy()[:20]  # first 20 token logits for clarity
temps  = [0.3, 0.7, 1.0, 1.5]

fig, axes = plt.subplots(1, len(temps), figsize=(14, 4), sharey=False)
for ax, t in zip(axes, temps):
    scaled = logits / t
    softmax = np.exp(scaled - scaled.max())
    softmax /= softmax.sum()
    ax.bar(range(len(softmax)), softmax, color='steelblue')
    ax.set_title(f'Temp = {t}', fontsize=11)
    ax.set_xlabel('Token index')
    if ax == axes[0]:
        ax.set_ylabel('Probability')
fig.suptitle('Effect of Temperature on Token Probability Distribution', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Decoding strategy comparison grouped bar chart (BLEU proxy) ---
strat_names = ['Greedy', 'Beam Search', 'Top-k', 'Top-p']
ref_tokens  = reference[0]
bleu_scores = []
for strat in ['greedy', 'beam_search', 'top_k', 'top_p']:
    gen = generate_text('Artificial intelligence will transform', strat)
    hyp = nltk.word_tokenize(gen.lower())
    bleu_scores.append(sentence_bleu([ref_tokens], hyp, smoothing_function=sf))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(strat_names, bleu_scores, color=sns.color_palette('Set2', 4), edgecolor='black')
ax.set_ylabel('BLEU Score')
ax.set_title('BLEU Score by Decoding Strategy')
for i, v in enumerate(bleu_scores):
    ax.text(i, v + 0.002, f'{v:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

# Section 8: Attention Visualisation

GPT-2 uses **multi-head self-attention**. By extracting the attention weights we can visualise which tokens each position attends to, providing insight into the model's internal reasoning.

In [ ]:
# --- Extract attention weights ---
input_text = 'The cat sat on the mat'
inputs = tokenizer(input_text, return_tensors='pt').to(device)

with torch.no_grad():
    outputs = base_model(**inputs, output_attentions=True)

attentions = outputs.attentions  # tuple of (batch, heads, seq, seq) per layer
print(f'Number of layers : {len(attentions)}')
print(f'Shape per layer  : {attentions[0].shape}  (batch, heads, seq_len, seq_len)')

In [ ]:
# --- Attention heatmap (layer 0, head 0) ---
layer_idx = 0
head_idx  = 0

attn_matrix = attentions[layer_idx][0, head_idx].cpu().numpy()
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    attn_matrix,
    xticklabels=tokens,
    yticklabels=tokens,
    cmap='viridis',
    ax=ax,
    linewidths=0.5,
)
ax.set_title(f'Attention Weights — Layer {layer_idx}, Head {head_idx}\nInput: "{input_text}"', fontsize=12)
ax.set_xlabel('Key tokens (attended to)')
ax.set_ylabel('Query tokens (attending)')
plt.tight_layout()
plt.show()

### Interpreting Attention Patterns

- **Diagonal patterns** indicate that each token primarily attends to itself.
- **Off-diagonal concentrations** show the model attending to earlier tokens in the context (GPT-2 uses causal/left-to-right attention, so a token cannot attend to future positions).
- Different heads often specialise — some may track syntactic dependencies (e.g. noun–verb agreement) while others focus on positional proximity.
- The lower-left triangle is zero because GPT-2 masks future tokens during self-attention.

# Section 9: Conclusions & Discussion

## Key Findings

1. **Decoding strategy matters significantly.**  
   Greedy decoding produces the most repetitive but grammatically clean text. Beam search improves coherence. Sampling-based methods (top-k, top-p) introduce greater lexical diversity at the cost of occasional incoherence.

2. **Temperature controls creativity vs. conservatism.**  
   Low temperatures (< 0.7) make the model more deterministic; high temperatures (> 1.2) increase randomness and may produce nonsensical output.

3. **Fine-tuning reduces perplexity.**  
   Even a single epoch of fine-tuning on WikiText-2 measurably improves perplexity on the validation set, demonstrating that domain adaptation is effective.

4. **Attention heads are interpretable.**  
   Distinct attention heads learn different relationships; some focus on neighbouring tokens, others on syntactic anchors.

## Limitations

- GPT-2 (small) is an older model. Modern LLMs (GPT-4, LLaMA-3, Mistral) significantly outperform it on all metrics.
- The fine-tuning here used only 500 training samples for speed; a full run would require the complete WikiText-2 training split.
- Perplexity is a limited metric — a low-perplexity model can still produce biased or factually incorrect text.

## Future Work

- Explore larger GPT-2 variants (medium, large, xl) or more modern models (GPT-Neo, OPT, Mistral).
- Implement RLHF (Reinforcement Learning from Human Feedback) for alignment.
- Add retrieval-augmented generation (RAG) to ground outputs in factual knowledge.
- Extend evaluations to TruthfulQA, HellaSwag, or other LLM benchmarks.

## Ethical Considerations

- **Bias:** GPT-2 was trained on web text and inherits societal biases present in that data.
- **Misinformation:** Language models can produce plausible-sounding but factually wrong content.
- **Misuse:** Autoregressive models can be misused for generating spam, phishing emails, or propaganda.
- Always apply content filtering and human oversight when deploying language models in production.

---

*ShadowFox AIML Internship — Advanced Track*